# Equity Premia Engine — v0.1
## Standard U.S. Factor Models: CAPM → FF3 → Carhart4 → FF5 → FF6

This notebook is the first **equity module** of a broader empirical risk-premia engine.

Its purpose is **not** to manufacture an equity timing strategy. The primary research questions are:

1. Which standard factor model best decomposes U.S. equity portfolio returns?
2. Which factors materially reduce pricing errors?
3. Which factors are redundant or spanned by others?
4. How stable are factor exposures and estimated prices of risk?
5. Do conclusions survive a leakage-safe out-of-sample attribution test?

The architecture deliberately mirrors the Rates Premia Engine:

\[
\text{test assets}
\rightarrow
\text{factor exposures } \beta
\rightarrow
\text{factor premia } \lambda
\rightarrow
\text{pricing errors}
\rightarrow
\text{model comparison}.
\]

### Models

- **CAPM**: \(MKT\)
- **FF3**: \(MKT, SMB, HML\)
- **Carhart4**: \(MKT, SMB, HML, MOM\)
- **FF5**: \(MKT, SMB, HML, RMW, CMA\)
- **FF6**: \(FF5 + MOM\)

A key implementation detail is preserved: **FF3/Carhart use the canonical FF3-family SMB, while FF5/FF6 use the canonical FF5-family SMB.** The two SMB constructions are not treated as interchangeable.

### Test assets

Two independent diagnostic universes are used:

- 25 value-weighted Size × Book-to-Market portfolios;
- 30 value-weighted industry portfolios.

The notebook also reports a pooled universe, but pooled tests should be interpreted as diagnostics rather than as a claim of cross-sectional independence.

### Data source

Kenneth R. French Data Library:
https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

**Data-methodology caveat.** Beginning with the January 2025 release, the U.S. French library moved from CRSP's legacy FIZ files to the CIZ flat-file format. This affects the construction mechanics of historical monthly returns (notably dividend timing). The notebook records the source date range but does not attempt to splice a legacy methodology internally.

## 0. Environment

In [ ]:
import sys
import subprocess
import importlib.util
from pathlib import Path

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "matplotlib": "matplotlib",
    "requests": "requests",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

import io
import re
import zipfile
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import requests
import scipy.stats as st
import statsmodels.api as sm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("Environment ready.")

## 1. Configuration

In [ ]:
START_DATE = "1963-07-31"
END_DATE = None

BETA_WINDOW = 60
MIN_FMB_ASSETS = 15
HAC_LAGS = 6
FMB_HAC_LAGS = 6

FRENCH_BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp"

FRENCH_FILES = {
    "FF3": "F-F_Research_Data_Factors_CSV.zip",
    "FF5": "F-F_Research_Data_5_Factors_2x3_CSV.zip",
    "MOM": "F-F_Momentum_Factor_CSV.zip",
    "SIZE_BM_25": "25_Portfolios_5x5_CSV.zip",
    "INDUSTRY_30": "30_Industry_Portfolios_CSV.zip",
}

CACHE_DIR = Path("equity_premia_cache")
CACHE_DIR.mkdir(exist_ok=True)

OUTPUT_DIR = Path("equity_premia_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print({
    "START_DATE": START_DATE,
    "BETA_WINDOW": BETA_WINDOW,
    "HAC_LAGS": HAC_LAGS,
})

## 2. Robust Kenneth French downloader and monthly-table parser

In [ ]:
_MONTHLY_ROW = re.compile(r"^\s*\d{6}\s*,")

def _download_zip_text(
    filename: str,
    cache_dir: Path = CACHE_DIR,
    timeout: int = 60,
) -> str:
    cache_path = cache_dir / filename.replace(".zip", ".txt")

    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8", errors="replace")

    url = f"{FRENCH_BASE}/{filename}"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (compatible; EmpiricalRiskPremiaEngine/0.1; "
            "+research-notebook)"
        )
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=timeout,
    )
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        csv_names = [
            name
            for name in zf.namelist()
            if name.lower().endswith(".csv")
        ]
        if not csv_names:
            raise ValueError(f"No CSV found inside {filename}")

        raw = zf.read(csv_names[0])

    text = raw.decode("utf-8", errors="replace")
    cache_path.write_text(text, encoding="utf-8")
    return text


def _find_monthly_block(
    text: str,
    anchor: str | None = None,
) -> tuple[str, list[str]]:
    lines = text.splitlines()
    scan_start = 0

    if anchor is not None:
        matches = [
            i for i, line in enumerate(lines)
            if anchor.lower() in line.lower()
        ]
        if not matches:
            raise ValueError(f"Anchor not found: {anchor}")
        scan_start = matches[0] + 1

    first_data_idx = None
    for i in range(scan_start, len(lines)):
        if _MONTHLY_ROW.match(lines[i]):
            first_data_idx = i
            break

    if first_data_idx is None:
        raise ValueError("Could not locate a YYYYMM monthly data block.")

    header_idx = None
    for i in range(first_data_idx - 1, scan_start - 2, -1):
        line = lines[i].strip()
        if line and "," in line:
            header_idx = i
            break

    if header_idx is None:
        raise ValueError("Could not locate the monthly table header.")

    rows = []
    i = first_data_idx
    while i < len(lines) and _MONTHLY_ROW.match(lines[i]):
        rows.append(lines[i])
        i += 1

    return lines[header_idx], rows


def load_french_monthly(
    filename: str,
    anchor: str | None = None,
) -> pd.DataFrame:
    text = _download_zip_text(filename)
    header, rows = _find_monthly_block(text, anchor=anchor)

    block = "\n".join([header, *rows])
    df = pd.read_csv(io.StringIO(block))

    df.columns = [
        str(c).strip()
        .replace("Mkt-RF", "MKT")
        .replace("Mkt_RF", "MKT")
        for c in df.columns
    ]

    first_col = df.columns[0]
    df = df.rename(columns={first_col: "YYYYMM"})
    df["YYYYMM"] = df["YYYYMM"].astype(str).str.strip()

    df.index = (
        pd.to_datetime(df["YYYYMM"], format="%Y%m")
        + pd.offsets.MonthEnd(0)
    )
    df = df.drop(columns=["YYYYMM"])

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.replace(
        [-99.99, -999.0, -999.99, -9999.0],
        np.nan,
    )

    df = df / 100.0
    df.index.name = "Date"

    return df.sort_index()


print("French downloader/parser ready.")

### Parser sanity logic

In [ ]:
_mock = """
metadata
Average Value Weighted Returns -- Monthly
,Asset A,Asset B
196307,1.00,2.00
196308,-1.00,3.00

Average Equal Weighted Returns -- Monthly
,Asset A,Asset B
196307,4.00,5.00
"""

_header, _rows = _find_monthly_block(
    _mock,
    anchor="Average Value Weighted Returns -- Monthly",
)

assert _header.strip() == ",Asset A,Asset B"
assert len(_rows) == 2

print("Monthly-table parser smoke test passed.")

## 3. Download canonical factor libraries

In [ ]:
ff3_raw = load_french_monthly(
    FRENCH_FILES["FF3"],
)

ff5_raw = load_french_monthly(
    FRENCH_FILES["FF5"],
)

mom_raw = load_french_monthly(
    FRENCH_FILES["MOM"],
)

mom_col = [
    c for c in mom_raw.columns
    if c.lower().replace(" ", "") in {"mom", "umd"}
]
if not mom_col:
    raise ValueError(
        f"Could not identify momentum column. Found: {list(mom_raw.columns)}"
    )

mom = mom_raw[[mom_col[0]]].rename(
    columns={mom_col[0]: "MOM"}
)

ff3 = ff3_raw.copy()
ff5 = ff5_raw.copy()

def _slice(df: pd.DataFrame) -> pd.DataFrame:
    out = df.loc[pd.Timestamp(START_DATE):].copy()
    if END_DATE is not None:
        out = out.loc[:pd.Timestamp(END_DATE)]
    return out

ff3 = _slice(ff3)
ff5 = _slice(ff5)
mom = _slice(mom)

print("FF3:", ff3.index.min().date(), "to", ff3.index.max().date(), ff3.shape)
print("FF5:", ff5.index.min().date(), "to", ff5.index.max().date(), ff5.shape)
print("MOM:", mom.index.min().date(), "to", mom.index.max().date(), mom.shape)

### Canonical-model registry

The model registry intentionally keeps the two Fama–French families separate:

\[
FF3/Carhart \neq \text{first three columns of FF5}.
\]

In particular, the FF5-family SMB construction aggregates size spreads across the book-to-market, profitability, and investment sorts.

In [ ]:
MODEL_REGISTRY = {
    "CAPM": {
        "family": "FF3",
        "factors": ["MKT"],
    },
    "FF3": {
        "family": "FF3",
        "factors": ["MKT", "SMB", "HML"],
    },
    "CARHART4": {
        "family": "FF3",
        "factors": ["MKT", "SMB", "HML", "MOM"],
    },
    "FF5": {
        "family": "FF5",
        "factors": ["MKT", "SMB", "HML", "RMW", "CMA"],
    },
    "FF6": {
        "family": "FF5",
        "factors": ["MKT", "SMB", "HML", "RMW", "CMA", "MOM"],
    },
}


def factor_frame_for_model(
    model_name: str,
) -> pd.DataFrame:
    spec = MODEL_REGISTRY[model_name]
    base = ff3.copy() if spec["family"] == "FF3" else ff5.copy()

    if "MOM" in spec["factors"]:
        base = base.join(mom, how="inner")

    cols = [*spec["factors"], "RF"]
    missing = [c for c in cols if c not in base.columns]
    if missing:
        raise ValueError(
            f"{model_name}: missing columns {missing}; available={list(base.columns)}"
        )

    return base[cols].dropna()


FACTORS_BY_MODEL = {
    model: factor_frame_for_model(model)
    for model in MODEL_REGISTRY
}

for model, frame in FACTORS_BY_MODEL.items():
    print(
        f"{model:10s} | {frame.index.min().date()} -> {frame.index.max().date()} "
        f"| {len(frame):4d} months | {MODEL_REGISTRY[model]['factors']}"
    )

## 4. Download test assets

In [ ]:
size_bm_25 = load_french_monthly(
    FRENCH_FILES["SIZE_BM_25"],
    anchor="Average Value Weighted Returns -- Monthly",
)

industry_30 = load_french_monthly(
    FRENCH_FILES["INDUSTRY_30"],
    anchor="Average Value Weighted Returns -- Monthly",
)

size_bm_25 = _slice(size_bm_25)
industry_30 = _slice(industry_30)

print("25 Size-B/M:", size_bm_25.index.min().date(), "to", size_bm_25.index.max().date(), size_bm_25.shape)
print("30 Industries:", industry_30.index.min().date(), "to", industry_30.index.max().date(), industry_30.shape)

display(size_bm_25.tail())
display(industry_30.tail())

## 5. Build excess-return universes

In [ ]:
rf_master = ff5["RF"].copy()

def excess_returns(
    raw_returns: pd.DataFrame,
    rf: pd.Series = rf_master,
) -> pd.DataFrame:
    joined = raw_returns.join(
        rf.rename("RF"),
        how="inner",
    )
    return joined[raw_returns.columns].sub(
        joined["RF"],
        axis=0,
    )

ASSET_UNIVERSES = {
    "SIZE_BM_25": excess_returns(size_bm_25),
    "INDUSTRY_30": excess_returns(industry_30),
}

pooled = pd.concat(
    [
        ASSET_UNIVERSES["SIZE_BM_25"].add_prefix("SBM_"),
        ASSET_UNIVERSES["INDUSTRY_30"].add_prefix("IND_"),
    ],
    axis=1,
    join="inner",
)

ASSET_UNIVERSES["POOLED_55"] = pooled

for name, data in ASSET_UNIVERSES.items():
    print(
        f"{name:12s} | {data.shape[1]:2d} assets | "
        f"{data.index.min().date()} -> {data.index.max().date()}"
    )

## 6. Descriptive diagnostics

In [ ]:
asset_summary = []

for universe_name, y in ASSET_UNIVERSES.items():
    corr = y.corr()
    mask = ~np.eye(y.shape[1], dtype=bool)

    asset_summary.append(
        {
            "universe": universe_name,
            "n_assets": y.shape[1],
            "n_months": len(y),
            "avg_mean_excess_ann": 12.0 * y.mean().mean(),
            "avg_ann_vol": np.sqrt(12.0) * y.std().mean(),
            "avg_pairwise_corr": corr.where(mask).stack().mean(),
        }
    )

asset_summary = pd.DataFrame(asset_summary).set_index("universe")
display(asset_summary)

In [ ]:
factor_union = (
    ff5[["MKT", "SMB", "HML", "RMW", "CMA", "RF"]]
    .join(mom, how="inner")
    .dropna()
)

factor_premia_summary = pd.DataFrame(
    {
        "mean_monthly": factor_union.drop(columns=["RF"]).mean(),
        "mean_ann_arithmetic": 12.0 * factor_union.drop(columns=["RF"]).mean(),
        "ann_vol": np.sqrt(12.0) * factor_union.drop(columns=["RF"]).std(),
    }
)

factor_premia_summary["historical_sharpe"] = (
    factor_premia_summary["mean_ann_arithmetic"]
    / factor_premia_summary["ann_vol"]
)

display(factor_premia_summary)

## 7. Time-series factor-model estimation

For each test asset \(i\),

\[
R^e_{i,t}
=
\alpha_i
+
\beta_i'F_t
+
\varepsilon_{i,t}.
\]

The economically important diagnostics are not only \(R^2\), but also:

- annualized pricing error \(12\alpha_i\);
- HAC \(t(\alpha_i)\);
- cross-asset RMSE of alpha;
- fraction of assets with significant residual alpha;
- Gibbons–Ross–Shanken joint alpha test.

In [ ]:
@dataclass
class TimeSeriesModelFit:
    asset_stats: pd.DataFrame
    betas: pd.DataFrame
    residuals: pd.DataFrame
    common_returns: pd.DataFrame
    common_factors: pd.DataFrame


def fit_time_series_model(
    asset_excess_returns: pd.DataFrame,
    model_name: str,
    hac_lags: int = HAC_LAGS,
) -> TimeSeriesModelFit:

    factor_data = FACTORS_BY_MODEL[model_name]
    factor_cols = MODEL_REGISTRY[model_name]["factors"]

    common = asset_excess_returns.join(
        factor_data[factor_cols],
        how="inner",
    ).dropna()

    y = common[asset_excess_returns.columns]
    f = common[factor_cols]

    stats_rows = []
    beta_rows = {}
    residuals = pd.DataFrame(
        index=common.index,
        columns=y.columns,
        dtype=float,
    )

    X = sm.add_constant(f, has_constant="add")

    for asset in y.columns:
        fit = sm.OLS(
            y[asset],
            X,
            missing="drop",
        ).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags},
        )

        alpha = float(fit.params["const"])
        alpha_t = float(fit.tvalues["const"])

        stats_rows.append(
            {
                "asset": asset,
                "alpha_monthly": alpha,
                "alpha_ann": 12.0 * alpha,
                "alpha_t_hac": alpha_t,
                "adj_r2": float(fit.rsquared_adj),
                "resid_ann_vol": np.sqrt(12.0) * float(np.std(fit.resid, ddof=1)),
                "nobs": int(fit.nobs),
            }
        )

        beta_rows[asset] = fit.params[factor_cols]
        residuals.loc[fit.resid.index, asset] = fit.resid

    asset_stats = pd.DataFrame(stats_rows).set_index("asset")
    betas = pd.DataFrame(beta_rows).T

    return TimeSeriesModelFit(
        asset_stats=asset_stats,
        betas=betas,
        residuals=residuals,
        common_returns=y,
        common_factors=f,
    )


print("Time-series estimator ready.")

## 8. Gibbons–Ross–Shanken joint-alpha test

In [ ]:
def grs_test(
    asset_excess_returns: pd.DataFrame,
    factor_returns: pd.DataFrame,
) -> dict:
    data = asset_excess_returns.join(
        factor_returns,
        how="inner",
    ).dropna()

    asset_cols = list(asset_excess_returns.columns)
    factor_cols = list(factor_returns.columns)

    Y = data[asset_cols].to_numpy(float)
    F = data[factor_cols].to_numpy(float)

    T, N = Y.shape
    K = F.shape[1]

    if T <= N + K:
        return {
            "grs": np.nan,
            "pvalue": np.nan,
            "df1": N,
            "df2": T - N - K,
        }

    X = np.column_stack([np.ones(T), F])
    coef = np.linalg.lstsq(X, Y, rcond=None)[0]

    alpha = coef[0, :]
    resid = Y - X @ coef

    sigma_e = resid.T @ resid / T

    mu_f = F.mean(axis=0)
    Fc = F - mu_f
    sigma_f = Fc.T @ Fc / T

    numerator = float(
        alpha
        @ np.linalg.pinv(sigma_e)
        @ alpha
    )

    denominator = float(
        1.0
        + mu_f
        @ np.linalg.pinv(sigma_f)
        @ mu_f
    )

    grs = (
        (T - N - K) / N
        * numerator
        / denominator
    )

    pvalue = st.f.sf(
        grs,
        N,
        T - N - K,
    )

    return {
        "grs": float(grs),
        "pvalue": float(pvalue),
        "df1": int(N),
        "df2": int(T - N - K),
    }


print("GRS test ready.")

## 9. Full-sample model horse race

In [ ]:
FULL_SAMPLE_FITS = {}
full_sample_rows = []

for universe_name, y in ASSET_UNIVERSES.items():
    for model_name in MODEL_REGISTRY:
        fit = fit_time_series_model(
            y,
            model_name,
        )
        FULL_SAMPLE_FITS[(universe_name, model_name)] = fit

        stats_df = fit.asset_stats
        factor_cols = MODEL_REGISTRY[model_name]["factors"]

        grs = grs_test(
            fit.common_returns,
            fit.common_factors[factor_cols],
        )

        full_sample_rows.append(
            {
                "universe": universe_name,
                "model": model_name,
                "n_factors": len(factor_cols),
                "mean_abs_alpha_ann": stats_df["alpha_ann"].abs().mean(),
                "rmse_alpha_ann": np.sqrt(np.mean(stats_df["alpha_ann"] ** 2)),
                "median_abs_alpha_ann": stats_df["alpha_ann"].abs().median(),
                "avg_adj_r2": stats_df["adj_r2"].mean(),
                "pct_alpha_sig_5pct": (stats_df["alpha_t_hac"].abs() > 1.96).mean(),
                "grs": grs["grs"],
                "grs_pvalue": grs["pvalue"],
            }
        )

full_sample_comparison = (
    pd.DataFrame(full_sample_rows)
    .set_index(["universe", "model"])
    .sort_index()
)

display(full_sample_comparison)

### Pricing-error comparison

In [ ]:
for universe_name in ASSET_UNIVERSES:
    table = (
        full_sample_comparison
        .loc[universe_name]
        [[
            "n_factors",
            "mean_abs_alpha_ann",
            "rmse_alpha_ann",
            "avg_adj_r2",
            "pct_alpha_sig_5pct",
            "grs_pvalue",
        ]]
        .sort_values("rmse_alpha_ann")
    )

    print(f"\n=== {universe_name} ===")
    display(table)

In [ ]:
for universe_name in ["SIZE_BM_25", "INDUSTRY_30"]:
    plot_df = (
        full_sample_comparison
        .loc[universe_name]
        .reset_index()
        .sort_values("n_factors")
    )

    ax = plot_df.plot(
        x="model",
        y="rmse_alpha_ann",
        kind="bar",
        figsize=(9, 4),
        legend=False,
    )
    ax.set_title(f"{universe_name}: annualized alpha RMSE")
    ax.set_ylabel("Alpha RMSE")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

## 10. Nested-model incremental diagnostics

The relevant question is not simply whether a larger model has a higher \(R^2\).
For nested or near-nested comparisons we ask whether the additional factors materially reduce asset-pricing errors.

The comparisons below are prespecified:

- CAPM → FF3
- FF3 → Carhart4
- FF3 → FF5
- FF5 → FF6

In [ ]:
NESTED_PAIRS = [
    ("CAPM", "FF3"),
    ("FF3", "CARHART4"),
    ("FF3", "FF5"),
    ("FF5", "FF6"),
]

nested_rows = []

for universe_name in ASSET_UNIVERSES:
    table = full_sample_comparison.loc[universe_name]

    for base_model, richer_model in NESTED_PAIRS:
        base = table.loc[base_model]
        rich = table.loc[richer_model]

        nested_rows.append(
            {
                "universe": universe_name,
                "base": base_model,
                "richer": richer_model,
                "delta_alpha_rmse_ann": (
                    rich["rmse_alpha_ann"]
                    - base["rmse_alpha_ann"]
                ),
                "pct_alpha_rmse_reduction": (
                    1.0
                    - rich["rmse_alpha_ann"] / base["rmse_alpha_ann"]
                ),
                "delta_avg_adj_r2": (
                    rich["avg_adj_r2"]
                    - base["avg_adj_r2"]
                ),
                "delta_pct_sig_alpha": (
                    rich["pct_alpha_sig_5pct"]
                    - base["pct_alpha_sig_5pct"]
                ),
            }
        )

nested_comparison = pd.DataFrame(nested_rows)
display(nested_comparison)

## 11. Factor spanning and redundancy

A factor can have a positive historical mean yet add little new pricing information if it is spanned by the other factors.

For a candidate factor \(g_t\),

\[
g_t = \alpha_g + b'F_t + u_t.
\]

A non-zero \(\alpha_g\) is evidence that the candidate is not fully replicated by the benchmark span.

In [ ]:
def spanning_regression(
    target: pd.Series,
    benchmark: pd.DataFrame,
    hac_lags: int = HAC_LAGS,
) -> dict:
    data = pd.concat(
        [
            target.rename("target"),
            benchmark,
        ],
        axis=1,
        join="inner",
    ).dropna()

    y = data["target"]
    X = sm.add_constant(
        data[benchmark.columns],
        has_constant="add",
    )

    fit = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": hac_lags},
    )

    return {
        "alpha_monthly": float(fit.params["const"]),
        "alpha_ann": 12.0 * float(fit.params["const"]),
        "alpha_t_hac": float(fit.tvalues["const"]),
        "adj_r2": float(fit.rsquared_adj),
        "nobs": int(fit.nobs),
    }


ff3_mom = ff3[["MKT", "SMB", "HML"]].join(mom, how="inner")
ff5_mom = ff5[["MKT", "SMB", "HML", "RMW", "CMA"]].join(mom, how="inner")

spanning_specs = {
    "MOM | FF3": (
        ff3_mom["MOM"],
        ff3_mom[["MKT", "SMB", "HML"]],
    ),
    "RMW | FF3": (
        ff5["RMW"],
        ff3[["MKT", "SMB", "HML"]],
    ),
    "CMA | FF3": (
        ff5["CMA"],
        ff3[["MKT", "SMB", "HML"]],
    ),
    "HML | MKT+SMB+RMW+CMA (FF5 family)": (
        ff5["HML"],
        ff5[["MKT", "SMB", "RMW", "CMA"]],
    ),
    "MOM | FF5": (
        ff5_mom["MOM"],
        ff5_mom[["MKT", "SMB", "HML", "RMW", "CMA"]],
    ),
}

spanning_rows = []

for label, (target, benchmark) in spanning_specs.items():
    row = spanning_regression(
        target,
        benchmark,
    )
    row["test"] = label
    spanning_rows.append(row)

spanning_table = (
    pd.DataFrame(spanning_rows)
    .set_index("test")
)

display(spanning_table)

## 12. Static cross-sectional prices of risk

For each model we also estimate the cross-sectional relation

\[
\bar R_i^e = \lambda_0 + \beta_i'\lambda + \eta_i,
\]

using full-sample time-series betas.

Because the Fama–French and Momentum factors are traded portfolios, their own historical means provide a natural benchmark for \(\lambda\). The cross-sectional estimates are therefore shown **alongside**, not treated as automatically superior to, the traded-factor means.

In [ ]:
def static_cross_sectional_pricing(
    fit: TimeSeriesModelFit,
    model_name: str,
) -> tuple[pd.Series, pd.Series, dict]:

    factor_cols = MODEL_REGISTRY[model_name]["factors"]

    mean_returns = fit.common_returns.mean()
    betas = fit.betas[factor_cols].loc[mean_returns.index]

    X = sm.add_constant(
        betas,
        has_constant="add",
    )

    cs_fit = sm.OLS(
        mean_returns,
        X,
    ).fit()

    lambdas = cs_fit.params.copy()
    errors = cs_fit.resid.copy()

    metrics = {
        "cs_r2": float(cs_fit.rsquared),
        "pricing_rmse_monthly": float(np.sqrt(np.mean(errors ** 2))),
        "pricing_rmse_ann": 12.0 * float(np.sqrt(np.mean(errors ** 2))),
        "mean_abs_pricing_error_ann": 12.0 * float(errors.abs().mean()),
    }

    return lambdas, errors, metrics


cs_rows = []
CS_LAMBDAS = {}

for universe_name in ["SIZE_BM_25", "INDUSTRY_30", "POOLED_55"]:
    for model_name in MODEL_REGISTRY:
        fit = FULL_SAMPLE_FITS[(universe_name, model_name)]
        lambdas, errors, metrics = static_cross_sectional_pricing(
            fit,
            model_name,
        )

        CS_LAMBDAS[(universe_name, model_name)] = lambdas

        cs_rows.append(
            {
                "universe": universe_name,
                "model": model_name,
                **metrics,
            }
        )

cs_comparison = (
    pd.DataFrame(cs_rows)
    .set_index(["universe", "model"])
)

display(cs_comparison)

In [ ]:
for universe_name in ["SIZE_BM_25", "INDUSTRY_30"]:
    print(f"\n=== {universe_name} ===")

    for model_name in MODEL_REGISTRY:
        factor_cols = MODEL_REGISTRY[model_name]["factors"]
        factor_frame = FACTORS_BY_MODEL[model_name]

        traded_means = factor_frame[factor_cols].mean()
        cs_lambda = CS_LAMBDAS[(universe_name, model_name)].drop("const")

        comp = pd.DataFrame(
            {
                "traded_factor_mean_monthly": traded_means,
                "cross_sectional_lambda_monthly": cs_lambda,
            }
        )

        print(f"\n{model_name}")
        display(comp)

## 13. Leakage-safe rolling betas

For date \(t\), betas are estimated using only the previous `BETA_WINDOW` months:

\[
\widehat\beta_{i,t-1}
=
\operatorname{OLS}\left(
R^e_{i,s},F_s
\right),
\qquad
s=t-W,\dots,t-1.
\]

These betas support two separate diagnostics:

1. **OOS factor attribution:** use realized \(F_t\) with lagged betas to explain \(R^e_{i,t}\);
2. **Fama–MacBeth:** use lagged betas in the month-\(t\) cross-section to estimate \(\lambda_t\).

Neither diagnostic is an investment timing rule.

In [ ]:
def rolling_beta_panel(
    asset_excess_returns: pd.DataFrame,
    model_name: str,
    window: int = BETA_WINDOW,
) -> dict[pd.Timestamp, pd.DataFrame]:

    factor_data = FACTORS_BY_MODEL[model_name]
    factor_cols = MODEL_REGISTRY[model_name]["factors"]

    common = asset_excess_returns.join(
        factor_data[factor_cols],
        how="inner",
    ).dropna()

    Y = common[asset_excess_returns.columns]
    F = common[factor_cols]

    beta_path = {}

    for j in range(window, len(common)):
        dt = common.index[j]

        train_idx = common.index[j - window:j]
        X_train = sm.add_constant(
            F.loc[train_idx],
            has_constant="add",
        ).to_numpy(float)

        Y_train = Y.loc[train_idx].to_numpy(float)

        coef = np.linalg.lstsq(
            X_train,
            Y_train,
            rcond=None,
        )[0]

        beta = pd.DataFrame(
            coef[1:, :].T,
            index=Y.columns,
            columns=factor_cols,
        )

        beta_path[dt] = beta

    return beta_path


print("Rolling-beta estimator ready.")

## 14. Out-of-sample factor attribution

The OOS decomposition at month \(t\) is

\[
\widehat R^e_{i,t,\text{factor}}
=
\widehat\beta_{i,t-1}'F_t.
\]

The residual

\[
u_{i,t}^{OOS}
=
R^e_{i,t}
-
\widehat\beta_{i,t-1}'F_t
\]

measures the component not explained by the model using only pre-\(t\) exposure estimates.

This is deliberately **not** a return forecast: \(F_t\) is contemporaneously realized. The test asks whether the factor representation generalizes out of sample.

In [ ]:
def oos_factor_attribution(
    asset_excess_returns: pd.DataFrame,
    model_name: str,
    window: int = BETA_WINDOW,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:

    factor_data = FACTORS_BY_MODEL[model_name]
    factor_cols = MODEL_REGISTRY[model_name]["factors"]

    common = asset_excess_returns.join(
        factor_data[factor_cols],
        how="inner",
    ).dropna()

    Y = common[asset_excess_returns.columns]
    F = common[factor_cols]

    beta_path = rolling_beta_panel(
        asset_excess_returns,
        model_name,
        window=window,
    )

    pred_rows = {}
    resid_rows = {}

    for dt, beta in beta_path.items():
        f_t = F.loc[dt, factor_cols]
        pred = beta.to_numpy(float) @ f_t.to_numpy(float)

        pred_s = pd.Series(
            pred,
            index=beta.index,
        )

        y_t = Y.loc[dt, beta.index]

        pred_rows[dt] = pred_s
        resid_rows[dt] = y_t - pred_s

    predictions = pd.DataFrame(pred_rows).T.sort_index()
    residuals = pd.DataFrame(resid_rows).T.sort_index()
    realized = Y.reindex(predictions.index)[predictions.columns]

    err = residuals.to_numpy(float).ravel()
    actual = realized.to_numpy(float).ravel()

    valid = np.isfinite(err) & np.isfinite(actual)
    err = err[valid]
    actual = actual[valid]

    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))

    denom = float(np.sum((actual - np.mean(actual)) ** 2))
    oos_var_explained = (
        1.0 - float(np.sum(err ** 2)) / denom
        if denom > 0
        else np.nan
    )

    residual_means = residuals.mean()

    metrics = {
        "n_oos_months": int(len(predictions)),
        "rmse_monthly": rmse,
        "rmse_ann_vol_equiv": np.sqrt(12.0) * rmse,
        "mae_monthly": mae,
        "oos_variance_explained": oos_var_explained,
        "mean_abs_residual_mean_ann": 12.0 * residual_means.abs().mean(),
        "rmse_residual_mean_ann": 12.0 * np.sqrt(np.mean(residual_means ** 2)),
    }

    return predictions, residuals, metrics


OOS_ATTRIBUTION = {}
oos_rows = []

for universe_name, y in ASSET_UNIVERSES.items():
    for model_name in MODEL_REGISTRY:
        predictions, residuals, metrics = oos_factor_attribution(
            y,
            model_name,
            window=BETA_WINDOW,
        )

        OOS_ATTRIBUTION[(universe_name, model_name)] = {
            "predictions": predictions,
            "residuals": residuals,
        }

        oos_rows.append(
            {
                "universe": universe_name,
                "model": model_name,
                "n_factors": len(MODEL_REGISTRY[model_name]["factors"]),
                **metrics,
            }
        )

oos_comparison = (
    pd.DataFrame(oos_rows)
    .set_index(["universe", "model"])
)

display(oos_comparison)

### OOS common-sample horse race

All models are next compared only on dates available to **every** candidate model within each universe. This avoids declaring one model superior merely because it is evaluated over an easier subperiod.

In [ ]:
common_oos_rows = []

for universe_name, y in ASSET_UNIVERSES.items():
    model_resids = {
        model: OOS_ATTRIBUTION[(universe_name, model)]["residuals"]
        for model in MODEL_REGISTRY
    }

    common_dates = None
    for resid in model_resids.values():
        dates = resid.dropna(how="all").index
        common_dates = (
            dates
            if common_dates is None
            else common_dates.intersection(dates)
        )

    for model_name, resid in model_resids.items():
        r = resid.reindex(common_dates)
        y_common = y.reindex(common_dates)[r.columns]

        err = r.to_numpy(float).ravel()
        actual = y_common.to_numpy(float).ravel()

        valid = np.isfinite(err) & np.isfinite(actual)
        err = err[valid]
        actual = actual[valid]

        rmse = np.sqrt(np.mean(err ** 2))
        mae = np.mean(np.abs(err))

        denom = np.sum((actual - np.mean(actual)) ** 2)
        r2 = (
            1.0 - np.sum(err ** 2) / denom
            if denom > 0
            else np.nan
        )

        residual_means = r.mean()

        common_oos_rows.append(
            {
                "universe": universe_name,
                "model": model_name,
                "common_oos_start": common_dates.min(),
                "common_oos_end": common_dates.max(),
                "n_common_months": len(common_dates),
                "rmse_monthly": rmse,
                "mae_monthly": mae,
                "oos_variance_explained": r2,
                "mean_abs_residual_mean_ann": 12.0 * residual_means.abs().mean(),
            }
        )

common_oos_comparison = (
    pd.DataFrame(common_oos_rows)
    .set_index(["universe", "model"])
)

for universe_name in ASSET_UNIVERSES:
    print(f"\n=== {universe_name}: common OOS sample ===")
    display(
        common_oos_comparison
        .loc[universe_name]
        .sort_values("rmse_monthly")
    )

## 15. Fama–MacBeth prices of risk with lagged rolling betas

In [ ]:
def fama_macbeth_with_lagged_betas(
    asset_excess_returns: pd.DataFrame,
    model_name: str,
    beta_window: int = BETA_WINDOW,
    hac_lags: int = FMB_HAC_LAGS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    factor_cols = MODEL_REGISTRY[model_name]["factors"]
    factor_data = FACTORS_BY_MODEL[model_name]

    common = asset_excess_returns.join(
        factor_data[factor_cols],
        how="inner",
    ).dropna()

    Y = common[asset_excess_returns.columns]

    beta_path = rolling_beta_panel(
        asset_excess_returns,
        model_name,
        window=beta_window,
    )

    lambda_rows = []
    diagnostics = []

    for dt, beta in beta_path.items():
        y_t = Y.loc[dt].reindex(beta.index)

        cs = beta.copy()
        cs["ret"] = y_t

        cs = cs.replace([np.inf, -np.inf], np.nan).dropna()

        if len(cs) < max(MIN_FMB_ASSETS, len(factor_cols) + 3):
            continue

        X = sm.add_constant(
            cs[factor_cols],
            has_constant="add",
        )

        fit = sm.OLS(
            cs["ret"],
            X,
        ).fit()

        row = fit.params.copy()
        row.name = dt
        lambda_rows.append(row)

        diagnostics.append(
            {
                "Date": dt,
                "cs_r2": float(fit.rsquared),
                "n_assets": int(fit.nobs),
                "beta_condition_number": float(np.linalg.cond(X.to_numpy(float))),
            }
        )

    lambda_ts = pd.DataFrame(lambda_rows).sort_index()
    diag = pd.DataFrame(diagnostics).set_index("Date").sort_index()

    inference_rows = []

    for col in lambda_ts.columns:
        series = lambda_ts[col].dropna()

        X = np.ones((len(series), 1))
        fit = sm.OLS(series.to_numpy(float), X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags},
        )

        inference_rows.append(
            {
                "parameter": col,
                "mean_monthly": float(fit.params[0]),
                "mean_ann": 12.0 * float(fit.params[0]),
                "t_hac": float(fit.tvalues[0]),
                "n_months": len(series),
            }
        )

    inference = pd.DataFrame(inference_rows).set_index("parameter")

    return lambda_ts, inference, diag


FMB_RESULTS = {}

for universe_name in ["SIZE_BM_25", "INDUSTRY_30", "POOLED_55"]:
    print(f"\n================ {universe_name} ================")

    for model_name in MODEL_REGISTRY:
        lambda_ts, inference, diag = fama_macbeth_with_lagged_betas(
            ASSET_UNIVERSES[universe_name],
            model_name,
        )

        FMB_RESULTS[(universe_name, model_name)] = {
            "lambda_ts": lambda_ts,
            "inference": inference,
            "diagnostics": diag,
        }

        print(f"\n{model_name}")
        display(inference)

## 16. Fama–MacBeth vs traded factor-premium means

In [ ]:
fmb_vs_traded_rows = []

for universe_name in ["SIZE_BM_25", "INDUSTRY_30", "POOLED_55"]:
    for model_name in MODEL_REGISTRY:
        factor_cols = MODEL_REGISTRY[model_name]["factors"]
        inference = FMB_RESULTS[(universe_name, model_name)]["inference"]
        factor_frame = FACTORS_BY_MODEL[model_name]

        for factor in factor_cols:
            fmb_vs_traded_rows.append(
                {
                    "universe": universe_name,
                    "model": model_name,
                    "factor": factor,
                    "fmb_mean_ann": inference.loc[factor, "mean_ann"],
                    "fmb_t_hac": inference.loc[factor, "t_hac"],
                    "traded_factor_mean_ann": 12.0 * factor_frame[factor].mean(),
                    "difference_ann": (
                        inference.loc[factor, "mean_ann"]
                        - 12.0 * factor_frame[factor].mean()
                    ),
                }
            )

fmb_vs_traded = (
    pd.DataFrame(fmb_vs_traded_rows)
    .set_index(["universe", "model", "factor"])
)

display(fmb_vs_traded)

## 17. Factor correlation and numerical-conditioning diagnostics

In [ ]:
conditioning_rows = []

for model_name, frame in FACTORS_BY_MODEL.items():
    factors = MODEL_REGISTRY[model_name]["factors"]
    F = frame[factors].dropna()

    z = (F - F.mean()) / F.std(ddof=1)
    cond = np.linalg.cond(z.to_numpy(float))

    corr = F.corr()
    offdiag = corr.where(
        ~np.eye(len(factors), dtype=bool)
    ).stack()

    conditioning_rows.append(
        {
            "model": model_name,
            "n_factors": len(factors),
            "standardized_condition_number": cond,
            "max_abs_pairwise_corr": offdiag.abs().max() if len(offdiag) else np.nan,
            "median_abs_pairwise_corr": offdiag.abs().median() if len(offdiag) else np.nan,
        }
    )

conditioning_table = (
    pd.DataFrame(conditioning_rows)
    .set_index("model")
)

display(conditioning_table)

In [ ]:
ff6_frame = FACTORS_BY_MODEL["FF6"][
    MODEL_REGISTRY["FF6"]["factors"]
]

display(ff6_frame.corr())

## 18. Exposure stability example

The engine is designed to decompose actual instruments later. Before moving to stock- or ETF-level assets, we inspect whether standard test-portfolio betas themselves are reasonably stable.

The chart below uses the 25 Size–B/M universe and FF6, with strictly lagged rolling betas.

In [ ]:
ff6_beta_path = rolling_beta_panel(
    ASSET_UNIVERSES["SIZE_BM_25"],
    "FF6",
    window=BETA_WINDOW,
)

sample_assets = list(ASSET_UNIVERSES["SIZE_BM_25"].columns[:3])

for asset in sample_assets:
    beta_ts = pd.DataFrame(
        {
            dt: beta.loc[asset]
            for dt, beta in ff6_beta_path.items()
        }
    ).T

    ax = beta_ts.plot(
        figsize=(11, 5),
    )
    ax.set_title(f"FF6 rolling betas — {asset}")
    ax.set_ylabel("Beta")
    plt.tight_layout()
    plt.show()

## 19. Transparent model leaderboards — no composite score

v0.1 deliberately avoids a proprietary weighted score.

A single scalar such as

\[
0.4\times RMSE + 0.3\times R^2 + \cdots
\]

would embed arbitrary preferences. Instead the notebook reports separate leaderboards for distinct research objectives.

### Risk decomposition
Prefer low **common-sample OOS residual RMSE** and high OOS variance explained.

### Asset-pricing fit
Prefer low alpha RMSE / mean absolute alpha and a less strongly rejected joint-alpha test.

### Parsimony / stability
Inspect factor count, conditioning, spanning, and rolling-beta stability.

Later versions may implement user-selectable decision rules, but the raw diagnostics remain primary.

In [ ]:
for universe_name in ["SIZE_BM_25", "INDUSTRY_30", "POOLED_55"]:
    print(f"\n================ {universe_name} ================")

    risk_view = (
        common_oos_comparison
        .loc[universe_name]
        [[
            "rmse_monthly",
            "oos_variance_explained",
            "mean_abs_residual_mean_ann",
        ]]
        .join(
            full_sample_comparison
            .loc[universe_name]
            [["n_factors"]],
        )
        .sort_values("rmse_monthly")
    )

    pricing_view = (
        full_sample_comparison
        .loc[universe_name]
        [[
            "n_factors",
            "rmse_alpha_ann",
            "mean_abs_alpha_ann",
            "avg_adj_r2",
            "pct_alpha_sig_5pct",
            "grs_pvalue",
        ]]
        .sort_values("rmse_alpha_ann")
    )

    print("\nRisk-decomposition leaderboard")
    display(risk_view)

    print("\nAsset-pricing-fit leaderboard")
    display(pricing_view)

## 20. Summary diagnostics

In [ ]:
summary_rows = []

for universe_name in ASSET_UNIVERSES:
    fs = full_sample_comparison.loc[universe_name]
    oos = common_oos_comparison.loc[universe_name]

    summary_rows.append(
        {
            "universe": universe_name,
            "best_full_sample_alpha_rmse_model": fs["rmse_alpha_ann"].idxmin(),
            "best_full_sample_alpha_rmse": fs["rmse_alpha_ann"].min(),
            "best_oos_attribution_model": oos["rmse_monthly"].idxmin(),
            "best_oos_rmse": oos["rmse_monthly"].min(),
            "best_oos_variance_explained_model": oos["oos_variance_explained"].idxmax(),
            "best_oos_variance_explained": oos["oos_variance_explained"].max(),
        }
    )

summary_table = (
    pd.DataFrame(summary_rows)
    .set_index("universe")
)

display(summary_table)

## 21. Export research tables

In [ ]:
full_sample_comparison.to_csv(
    OUTPUT_DIR / "full_sample_model_comparison.csv"
)

nested_comparison.to_csv(
    OUTPUT_DIR / "nested_model_incremental_diagnostics.csv",
    index=False,
)

spanning_table.to_csv(
    OUTPUT_DIR / "factor_spanning_tests.csv"
)

cs_comparison.to_csv(
    OUTPUT_DIR / "static_cross_sectional_pricing.csv"
)

oos_comparison.to_csv(
    OUTPUT_DIR / "oos_attribution_model_comparison.csv"
)

common_oos_comparison.to_csv(
    OUTPUT_DIR / "common_oos_model_comparison.csv"
)

fmb_vs_traded.to_csv(
    OUTPUT_DIR / "fmb_vs_traded_factor_premia.csv"
)

conditioning_table.to_csv(
    OUTPUT_DIR / "factor_conditioning.csv"
)

summary_table.to_csv(
    OUTPUT_DIR / "summary_table.csv"
)

print("Research tables exported to:", OUTPUT_DIR.resolve())

# v0.1 interpretation guide

This notebook should be read in the following order:

1. **Does adding factors reduce alpha pricing errors?**
2. **Does the richer model generalize in the leakage-safe OOS attribution test?**
3. **Are the added factors genuinely incremental in spanning regressions?**
4. **Are cross-sectional prices of risk broadly consistent with traded factor premia?**
5. **Are conclusions stable across Size–B/M and Industry test assets?**
6. **Does model complexity create conditioning or unstable exposures?**

No model should be declared "best" because it wins one backtest metric.

## Planned extensions

### v0.2 — broader published factor families
- AQR BAB
- AQR QMJ
- q-factor / q5 family where a clean reproducible data source is available
- richer spanning tests

### v0.3 — proprietary defensive-factor research
- stock-level LowVol / BAV / BAC construction
- incremental LowVol information beyond BAB
- candidate augmented BAB representation

### later — conditional premia and integration
- \(p(\lambda_{t+H}\mid\mathcal F_t)\)
- regime engine integration
- scenario-engine integration
- instrument-level decomposition for stocks / ETFs / portfolios
- portfolio-construction use only after factor-premium validation